# Planars — Annotation Status

Shows the current span results based on your filled annotation sheets.

**No installation required. You only need:**
- A Google account with access to the shared planars Drive folder
- The path to that folder (ask the project coordinator if unsure)

---

**Instructions:**
1. Click the folder icon in the cell below and set `DRIVE_FOLDER_PATH` to your planars folder
2. From the menu: **Runtime → Run all**
3. The chart will appear at the bottom of the page

In [ ]:
# ❗ Set this to your planars Drive folder path
DRIVE_FOLDER_PATH = '/content/drive/MyDrive/planars \u2014 stan1293'

In [ ]:
# Step 1 — Install planars
# After this cell, go to Runtime → Restart session, then run from Step 2 onward.
!pip install -q --upgrade --pre planars gspread google-auth
import importlib.metadata
print(f"planars {importlib.metadata.version("planars")} ready — now restart the runtime")

In [ ]:
# Setup (runs automatically — nothing to change here)

from google.colab import auth, drive as gdrive
auth.authenticate_user()
gdrive.mount('/content/drive')

import json, gspread
from pathlib import Path
from google.auth import default
from planars.charts import collect_all_spans_from_sheets, domain_chart

creds, _ = default()
gc = gspread.authorize(creds)

folder = Path(DRIVE_FOLDER_PATH)
manifest_files = sorted(folder.glob('manifest_*.json'))
if not manifest_files:
    raise FileNotFoundError(
        f'No manifest file found in: {DRIVE_FOLDER_PATH}\n'
        'Please check that DRIVE_FOLDER_PATH is correct.'
    )

manifest = {}
for p in manifest_files:
    lang_id = p.stem.replace('manifest_', '')
    manifest[lang_id] = json.loads(p.read_text())

print(f'Connected. Loaded data for: {", ".join(manifest.keys())}')
print('Computing spans...')

_result = collect_all_spans_from_sheets(gc, manifest)
if len(_result) == 2:
    df, lang_meta = _result
else:
    # planars <0.1.0a3 — wrap old 3-tuple into lang_meta shape
    df, _kp, _p2n = _result
    lang_meta = {list(manifest.keys())[0]: {"keystone_pos": _kp, "pos_to_name": _p2n}}
lang_id, meta = next(iter(lang_meta.items()))
print(f'Done. {len(df)} spans computed for {lang_id}.')

In [ ]:
# Domain chart — hover over segments for details
domain_chart(df, meta["keystone_pos"], meta["pos_to_name"])